# ML Zoomcamp 2026 — Homework 4: Evaluation Metrics for Classification

Dataset: Course Lead Scoring (`course_lead_scoring_2026.csv`)  
Target variable: `converted`

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

%matplotlib inline

## Data Preparation

* Check for missing values
* Fill missing categorical values with `'NA'` and numerical values with `0.0`
* Split the data into 60% train, 20% validation, and 20% test sets (`random_state=1`)

In [ ]:
df = pd.read_csv('course_lead_scoring_2026.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

print('Missing values before imputation:')
print(df[categorical + numerical].isnull().sum())

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

print('\nMissing values after imputation:')
print(df[categorical + numerical].isnull().sum())

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)

## Question 1: ROC AUC feature importance

Which numerical variable (among `lead_score`, `number_of_courses_viewed`, `interaction_count`, `annual_income`) has the highest AUC on the training dataset?

In [ ]:
feature_auc_scores = {}

for feature in numerical:
    auc = roc_auc_score(y_train, df_train[feature])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[feature])
    feature_auc_scores[feature] = auc
    print(f'{feature:25s}: {auc:.4f}')

best_feature = max(feature_auc_scores, key=feature_auc_scores.get)
print('\nHighest AUC numerical feature:', best_feature)

**Answer Q1:** `lead_score` (AUC = `0.7882`).

## Question 2: Training the model

Apply one-hot encoding using `DictVectorizer(sparse=False)` and train `LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)`. What is the AUC of this model on the validation dataset (rounded to 3 digits)?

In [ ]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

y_pred = model.predict_proba(X_val)[:, 1]
validation_auc = roc_auc_score(y_val, y_pred)

print(f'Validation AUC: {validation_auc:.6f}')
print(f'Validation AUC (rounded to 3 digits): {round(validation_auc, 3):.3f}')

**Answer Q2:** `0.732` (exact validation AUC = `0.731822`).

## Question 3: Precision and Recall

Evaluate the model on all thresholds from `0.0` to `1.0` with step `0.01`. At which threshold do precision and recall curves intersect?

In [ ]:
scores = []
thresholds = np.linspace(0, 1, 101)

for t in thresholds:
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)

    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)

    tp = (predict_positive & actual_positive).sum()
    tn = (predict_negative & actual_negative).sum()
    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()

    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    scores.append((t, tp, fp, fn, tn, p, r, f1))

columns = ['threshold', 'tp', 'fp', 'fn', 'tn', 'p', 'r', 'f1']
df_scores = pd.DataFrame(scores, columns=columns)

df_valid_pr = df_scores[(df_scores.p > 0) | (df_scores.r > 0)].copy()
df_valid_pr['pr_diff'] = (df_valid_pr.p - df_valid_pr.r).abs()

best_pr_row = df_valid_pr.loc[df_valid_pr['pr_diff'].idxmin()]
print(f"Intersection threshold: {best_pr_row['threshold']:.2f} (p={best_pr_row['p']:.4f}, r={best_pr_row['r']:.4f}, |p - r|={best_pr_row['pr_diff']:.4f})")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(df_scores.threshold, df_scores.p, label='Precision')
plt.plot(df_scores.threshold, df_scores.r, label='Recall')
plt.axvline(best_pr_row['threshold'], color='grey', linestyle='--', label=f"Threshold = {best_pr_row['threshold']:.2f}")
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs. Threshold')
plt.legend()
plt.show()

**Answer Q3:** `0.63` (where precision = `0.7328`, recall = `0.7253`, and `|p - r| = 0.0075`).

## Question 4: F1 score

Compute $F_1 = 2 \cdot \frac{P \cdot R}{P + R}$ for all thresholds from `0.0` to `1.0` with increment `0.01`. At which threshold is $F_1$ maximal?

In [ ]:
best_f1_row = df_scores.loc[df_scores['f1'].idxmax()]
print(f"Max F1 threshold: {best_f1_row['threshold']:.2f} (F1={best_f1_row['f1']:.4f}, p={best_f1_row['p']:.4f}, r={best_f1_row['r']:.4f})")

plt.figure(figsize=(6, 4))
plt.plot(df_scores.threshold, df_scores.f1, label='F1 score')
plt.axvline(best_f1_row['threshold'], color='grey', linestyle='--', label=f"Threshold = {best_f1_row['threshold']:.2f}")
plt.xlabel('Threshold')
plt.ylabel('F1')
plt.title('F1 Score vs. Threshold')
plt.legend()
plt.show()

**Answer Q4:** `0.41` (maximal $F_1 = 0.7576$).

## Question 5: 5-Fold CV

Use `KFold(n_splits=5, shuffle=True, random_state=1)` to evaluate `LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)` on `df_full_train`. How large is the standard deviation of the scores across different folds?

In [ ]:
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X_train, y_train)

    return dv, model


def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=1)
fold_scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_fold_train = df_full_train.iloc[train_idx]
    df_fold_val = df_full_train.iloc[val_idx]

    y_fold_train = df_fold_train.converted.values
    y_fold_val = df_fold_val.converted.values

    fold_dv, fold_model = train(df_fold_train, y_fold_train, C=1.0)
    y_fold_pred = predict(df_fold_val, fold_dv, fold_model)

    fold_auc = roc_auc_score(y_fold_val, y_fold_pred)
    fold_scores.append(fold_auc)

print('Fold AUC scores:', [round(score, 4) for score in fold_scores])
print(f'Mean AUC: {np.mean(fold_scores):.4f}')
print(f'Std AUC : {np.std(fold_scores):.4f} (rounded to 3 digits: {round(np.std(fold_scores), 3):.3f})')

**Answer Q5:** `0.007` (exact standard deviation across the 5 folds = `0.0069`).

## Question 6: Hyperparameter Tuning

Iterate over `C` in `[0.000001, 0.001, 1]` using the same 5-fold split, and compute the mean and standard deviation of the AUC scores (rounded to 3 decimal digits). Which `C` leads to the best mean score?

In [ ]:
for C in [0.000001, 0.001, 1]:
    fold_scores = []

    for train_idx, val_idx in kfold.split(df_full_train):
        df_fold_train = df_full_train.iloc[train_idx]
        df_fold_val = df_full_train.iloc[val_idx]

        y_fold_train = df_fold_train.converted.values
        y_fold_val = df_fold_val.converted.values

        fold_dv, fold_model = train(df_fold_train, y_fold_train, C=C)
        y_fold_pred = predict(df_fold_val, fold_dv, fold_model)

        fold_auc = roc_auc_score(y_fold_val, y_fold_pred)
        fold_scores.append(fold_auc)

    print('C=%s %.3f +- %.3f' % (C, np.mean(fold_scores), np.std(fold_scores)))

**Answer Q6:** `0.001` (`0.749 +- 0.010`, highest mean AUC among the candidates).

## Summary of Answers

| Question | Topic | Result | Selected Option |
| :--- | :--- | :--- | :--- |
| **Q1** | Highest training ROC AUC numerical feature | `lead_score` (`0.7882`) | `lead_score` |
| **Q2** | Validation ROC AUC (`C=1.0`, rounded to 3 digits) | `0.731822` | `0.732` |
| **Q3** | Threshold where Precision and Recall curves intersect | `0.63` (`\|p - r\| = 0.0075`) | `0.63` |
| **Q4** | Threshold where $F_1$ score is maximal | `0.41` ($F_1 = 0.7576$) | `0.41` |
| **Q5** | 5-Fold CV ROC AUC standard deviation (`C=1.0`) | `0.0069` | `0.007` |
| **Q6** | Best `C` from 5-Fold CV hyperparameter tuning | `C=0.001` (`0.749 +- 0.010`) | `0.001` |